In [1]:
import pandas as pd
import numpy as np
from rapidfuzz import process

In [2]:
# we will use rapidfuzz's process module to match names for joining datasets
# metacritic and steam game titles may have slight differences in naming conventions
# process.extractOne helps find the best match from a list of choices
# rapidfuzz is much faster than fuzzywuzzy

In [3]:
meta = pd.read_csv("../data/metacritic_Toppc_games.csv")
steam = pd.read_csv("../data/steam_spy_data.csv")

matches = []

for name in meta['Name']:
    match, score, _ = process.extractOne(name, steam['name'])
    matches.append((name, match, score))

matches_df = pd.DataFrame(matches, columns=['metacritic_name', 'steam_name', 'similarity_score'])

matches_df = matches_df.sort_values(by='similarity_score', ascending=False)

In [4]:
matches_df[matches_df['similarity_score'].between(90, 92)]

,metacritic_name,steam_name,similarity_score
4619,Knights of Pen & Paper 2,Knights of Pen and Paper 2,92.000000
5632,Secrets of Raetikon,Secrets of Rætikon,91.891892
1465,TrackMania 2 Canyon,TrackMania² Canyon,91.891892
3674,The Golf Club 2019 featuring PGA Tour,The Golf Club 2019 featuring PGA TOUR,91.891892
591,The Binding of Isaac: Afterbirth,The Binding of Isaac: Rebirth,91.803279
...,...,...,...
2303,Dawn of Discovery: Venice,Dawn,90.000000
2304,Destiny 2: Shadowkeep,Destiny 2,90.000000
2305,Date Everything!,Everything,90.000000
2306,Dungelot: Shattered Lands,Shatter,90.000000


In [5]:
matches_df[matches_df['similarity_score'] >= 90].sample(50)

,metacritic_name,steam_name,similarity_score
59,Out of the Park Baseball 17,Out of the Park Baseball 19,96.296296
396,Total War: WARHAMMER II,Total War: WARHAMMER II,100.000000
148,Brothers: A Tale of Two Sons,Brothers: A Tale of Two Sons Remake,95.000000
4180,Lake,Lake,100.000000
2250,Medal of Honor: Airborne,Medal of Honor: Airborne,100.000000
4467,The Hong Kong Massacre,The Hong Kong Massacre,100.000000
3098,A Vampyre Story,Vampyr,90.000000
240,Mafia,Mafia,100.000000
4379,Ultima Online: Third Dawn,Dawn,90.000000
4916,Need for Speed: Undercover,Need for Speed Undercover,98.039216


In [6]:
# we found that a similarity score of 90 is a good threshold for matching names
# after manually checking a sample of matches at this score
# we found there is a 90% accuracy rate of correct matches at this threshold
# so we will keep matches with a similarity score of 90 or higher for joining datasets

high_confidence_matches = matches_df[matches_df['similarity_score'] >= 90]

In [7]:
high_confidence_matches.head(5)

,metacritic_name,steam_name,similarity_score
6292,Stormgate,Stormgate,100.0
6271,Rogue Warrior,Rogue Warrior,100.0
6264,POSTAL 4: No Regerts,POSTAL 4: No Regerts,100.0
31,Red Dead Redemption 2,Red Dead Redemption 2,100.0
29,God of War,God of War,100.0


In [8]:
high_confidence_matches.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3713 entries, 6292 to 19
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   metacritic_name   3713 non-null   object 
 1   steam_name        3713 non-null   object 
 2   similarity_score  3713 non-null   float64
dtypes: float64(1), object(2)
memory usage: 116.0+ KB


In [9]:
high_confidence_matches.to_csv("../data/matched_names.csv", index=False)

# we now have a table with names from metacritic data and similir name in steam data we can use later to join on